In [1]:
import numpy as np
from src.LWE import *
from src.RLWE import *
from src.GSW_utils import *
from src.GSW import *
from src.bootstrap import *
from src.polynomial import *

In [2]:
# setting all the configs and generating all the required keys
lweconfig = LWEConfig(p=1<<4, q=1<<32, noise_level=2**-20, dimension=1024)
config = RLWEConfig(N=1024, sigma=2**(-24), p = 16, q=2**32)
gsw_config = GSWConfig(config, 2**8)
lwe_key = LWEEncryptionKey(config=lweconfig)
lwe_key.generate_key()
rlwe_key = lwe_to_rlwe_key(lwe_key, config)
# get the gsw key
gsw_key = GSWEncryptionKey(gsw_config)
gsw_key.generate_key_from_rlwe_key(rlwe_key)
# get the bootstrap key
boostrap_key = BootstrapKey(gsw_config)
boostrap_key.generate_bootstrap_key(lwe_key=lwe_key, gsw_key=gsw_key)

In [3]:
# generating the polynomial
tx_coeffs = np.zeros(config.N, dtype=np.int32)
tx_coeffs[config.N//2:] = -1
tx = Polynomial(N=config.N, coeffs=tx_coeffs)
# encoding tx
encode_multiplier = encode_plaintext(1, lweconfig)
# generating a trivial RLWE polynomial for tx
tx_rlwe = create_trivial_rlwe_ciphertext(tx.polynomial_const_multiply(encode_multiplier.m), config)

In [4]:
tx_coeffs

array([ 0,  0,  0, ..., -1, -1, -1], shape=(1024,), dtype=int32)

In [6]:
def sign_bootstrap(i: LWECiphertext, bsk: BootstrapKey, tx_cipher: RLWECiphertext):
    '''
        This function performs PBS which evaluates the sign function. This will return an LWE ciphertext encrypting 0 if the input was negative or 0, 1 otherwise.
    '''
    # need to implement coeff(0, blindrotate(i, tx_cipher))
    # blind rotate tx_cipher
    rotated_tx = bsk.blindrotate(tx_cipher, i)
    # extract the 0th coefficient
    coeff_lwe = extract_sample(0, rotated_tx)
    return coeff_lwe

In [32]:
# getting the sign for an encrypted number
n1 = encode_plaintext(-4, lweconfig)
n1_lwe = lwe_key.encrypt(n1)

In [33]:
sign_n1 = sign_bootstrap(n1_lwe, boostrap_key, tx_rlwe)

In [34]:
# decrypting sign_n1
print(lwe_key.decrypt(sign_n1))

0


### Using the sign operation to calculate the max between two numbers
Assuming that sign is a {0, 1} function, we can use the CMux operation with the result of sign being the selector bit. However, the selector bit has to be a GSW encrypted ciphertext, while the result of the sign operation is an LWE ciphertext. Converting an LWE ciphertext to GSW is, apparantly, non-trivial (based on what Ive seen, it might be easy though.<br>
Another way of doing this can also be just implementing the max function as a PBS operation. However, the LUT for that is a little difficult and therefore, I will be using the TFHE-rs library directly to do that.